In [2]:
%%capture
!pip install --upgrade transformers==4.41.2 sentence-transformers==3.0.1 gensim==4.3.2 scikit-learn==1.5.0 accelerate==0.31.0 peft==0.11.1 scipy==1.10.1 numpy==1.26.4

without understanding tokens and embeddings we cannot understand how LLMs works see fig 2-1

LLM TOKENIZATION

the model generates its response in one token at a time, but tokens arent only the outputs but also the input which the model sees, a text form is broken into tokens before sending it to the model

HOW tokenizers prepare the inputs for LLMs

if we see  a high level view LLMS take an input prompt and then generate a response
but thats not what happens inside, the input prompt has to go through a tokenizer which breaks the input into pieces , see fig 2-3

to further understand this lets download an LLM and see how can we tokenize the input prompt



In [3]:
from transformers import AutoModelForCausalLM, AutoTokenizer

# Load model and tokenizer
model = AutoModelForCausalLM.from_pretrained(
    "microsoft/Phi-3-mini-4k-instruct",
    device_map="cuda",
    torch_dtype="auto",
    trust_remote_code=False,
)
tokenizer = AutoTokenizer.from_pretrained("microsoft/Phi-3-mini-4k-instruct")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/967 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/16.5k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

...

tokenizer_config.json:   0%|          | 0.00/3.44k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.94M [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/306 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/599 [00:00<?, ?B/s]

we will first write our prompt then tokenize it and give the tokens to the model, now we will tell the model to generate only 20 new tokens

In [4]:
prompt = "Write an email apologizing to Sarah for the tragic gardening mishap. Explain how it happened.<|assistant|>"

# Tokenize the input prompt
input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to("cuda")

# Generate the text
generation_output = model.generate(
input_ids=input_ids,
max_new_tokens=20

)
# Print the output
print(tokenizer.decode(generation_output[0]))

[transformers] The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Write an email apologizing to Sarah for the tragic gardening mishap. Explain how it happened.<|assistant|> Subject: Heartfelt Apologies for the Gardening Mishap


Dear


everything after subject: is the modles 20 tokens output, the model didnt actually got the raw prompt but the tokenizers took the raw prompt and returned the information needed by the model in the form of input_ids variable, which the model used as its input

In [5]:
print(input_ids)

tensor([[14350,   385,  4876, 27746,  5281,   304, 19235,   363,   278, 25305,
           293, 16423,   292,   286,   728,   481, 29889, 12027,  7420,   920,
           372,  9559, 29889, 32001]], device='cuda:0')


this shows the LLMs respond to a series of numbers as shown in fig 2-4, each one is unique id of the specific token, the token can be a word, a char, or part of the word, these ids reference a table inside the tokenizer which contains all the tokens it knows

if we want to see the actual words which these ids hold we can use the decode method by tokenizer

In [6]:
for id in input_ids[0]:
  print(tokenizer.decode(id))

Write
an
email
apolog
izing
to
Sarah
for
the
trag
ic
garden
ing
m
ish
ap
.
Exp
lain
how
it
happened
.
<|assistant|>


some tokens are complete words while some are part of words, punctuation char are their own tokens

there are no tokens for spaces and partial tokens like "izing" and "ic" have a special hidden char at their beginning which says they are connected with token that precedes them in text, tokens without that special char are assumed to have a space before them

if we move towards the output side we can also see the tokens generated by model for outputs using generation_output variable, it will show us the input tokens as well as the output tokens

In [7]:
print(generation_output)

tensor([[14350,   385,  4876, 27746,  5281,   304, 19235,   363,   278, 25305,
           293, 16423,   292,   286,   728,   481, 29889, 12027,  7420,   920,
           372,  9559, 29889, 32001,  3323,   622, 29901, 17778, 29888,  2152,
          6225, 11763,   363,   278, 19906,   292,   341,   728,   481,    13,
            13,    13, 29928,   799]], device='cuda:0')


after 32001 all the tokens are of output, we can see the actual text for output side too using tokenizers decode method

In [8]:
print(tokenizer.decode(3323))
print(tokenizer.decode(622))
print(tokenizer.decode([3323, 622]))
print(tokenizer.decode(29901))

Sub
ject
Subject
:


HOW DOES THE TOKENIZER BREAKDOWNS THE TEXT ?


it follows three major factors for this :

1. when we design the model we can choose the type of tokenization we want there are several methods like BYTE PAIR ENCODING (BPE) used in GPT models and WordPiece used in BERT, these methods are good at what they are designed to do which is to optimize an efficient set of tokens to represent a text dataset, but they do that in differnet ways

2. after deciding the method we neeed to choose a number of tokenizer design choices like vocab size and which special tokens to use

3. the tokenizer needs to be trained on the specific dataset to make the best vocab which it can use to represent the dataset, and even if we use the same params and methods a tokenizer trained on the english dataset will perform differently then the one trained on the code dataset or multiliinguial text dataset

tokenizers are also used during output gen, when the model returns an output token id, tokenizer is used to return the word associated with it as seen in fig 2-5

BYTE, CHAR, SUNWORDS and WORDS TOKENS


The tokenization way showed above is subword tokenization it is the most commonly used tokenization scheme, there 4 in total as shown in fig 2-6

1. word tokens
it was a common approach with methods like word2vec but now its becoming less popular in NLP, but they are still used in recommendation systems
there is one problem with word tokenizer, it cannot deal with new words that enter the dataset after the tokenizer is trained, and the vocabulary formed also have a lot of tokens for the words which have very minimum differences like apologize, apology, apologetic
this challange was solved by subword tokenization, it have a token for the apalog and then the suffix tokens like -y, -ize, -ogy, -etic, that are common with many other tokens as well this results in an efficient and more expressive vocab

2. subword tokens
this method contains partial words and full words as discussed arlier, it also have a benefit of represeting new words by breaking the new token into smaller chars, which will eventually be already present in the vocab

3. character tokens
this method can also deal with new words as it has raw letters to fall back on, this makes it robust and easier to tokenize but the modelling becomes more difficult for eg : if we use subword tokenization it will save "cat" as single token, but a model using char level tokens need to model the information to spell "c-a-t" while modelling the rest of the input too
and sub words tokens also have one more advantage, it can fit more text within the limited context length of a transformer model, subword tokens often average 3 chars per token

4. byte tokens
this method breaks down tokens into individual bytes that are used to represent the unicode chars, this method can be a competitive one in multilingual scenarios

some subword tokenizer methods also includes bytes as tokens in their vocab, they use it when they encounter chars they cant represent otherwise, the GPT 2 and Roberta tokenizers do this but this doesnt make them a tokenization free byte level tokenizers as they only represent a subset using this

Comparing Trained LLM Tokenizers

we will compare some tokenizers now, the newer tokenizers have changed their behvior to improve models performance, further we will see how code generation models or specialized models need specialized tokenizers

In [ ]:
text = """
English and CAPITALIZATION
🎵鸟
show_tokens False None elif == >= else: two tabs:" " Three tabs:" "
12.0*50=600
"""